In [23]:
from functools import partial

from pep_compass.local_enumeration.mutation.utils import get_mutations_from_s_u_standard
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import (
    HydrAMPEncoderDecoder,
)
from pep_compass.models.encoder_decoder.utils import decoder_jacobian

In [24]:
# plot_heatmap
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def plot_heatmap(
    matrix,
    title="Heatmap",
    xlabel="X",
    ylabel="Y",
    cmap="viridis",
    center=0,
    figsize=(10, 8),
    xticklabels=None,
    yticklabels=None,
    original_sequence=None,
    alphabet=None,
):
    """
    Plot a heatmap with optional tick labels.

    Args:
        matrix: 2D array to plot
        title: plot title
        xlabel: x-axis label
        ylabel: y-axis label
        cmap: colormap
        center: center value for colormap
        figsize: figure size tuple
        xticklabels: list of x-axis tick labels (or None for default)
        yticklabels: list of y-axis tick labels (or None for default)
        original_sequence: optional sequence string to mark original AAs with black squares
        alphabet: optional list of amino acids matching xticklabels order
    """
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        matrix,
        cmap=cmap,
        center=center,
        xticklabels=xticklabels if xticklabels is not None else "auto",
        yticklabels=yticklabels if yticklabels is not None else "auto",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Mark original amino acids with black squares
    if original_sequence is not None and alphabet is not None:
        for pos, aa in enumerate(original_sequence):
            try:
                # Find the column index for this amino acid in the alphabet
                aa_col = alphabet.index(aa)
                # Draw a black filled square at this position
                # In heatmap coordinates: (x, y) where x is column, y is row
                # Each cell is 1x1, centered at (col + 0.5, row + 0.5)
                rect = patches.Rectangle(
                    (aa_col, pos),  # bottom-left corner
                    1,  # width
                    1,  # height
                    fill=True,
                    facecolor="black",
                    edgecolor="black",
                    linewidth=0,
                    transform=ax.transData,
                )
                ax.add_patch(rect)
            except ValueError:
                # Amino acid not found in alphabet, skip
                pass

    plt.show()

In [25]:
hydramp = HydrAMPEncoderDecoder(jacobian_eps=1e-6, field_eps=1e-6)

In [26]:
seq = "IDYTCQIAKTIYGILGIKIWIFQKI"

In [27]:
z = hydramp.encode_peptides([seq])

In [28]:
jac_fn = partial(
    decoder_jacobian,
    decoder_forward=hydramp.decoder_forward,
    jacobian_fn_mode="approx",
    jacobian_fn_kwargs={"jacobian_eps": 1e-6},
)

In [29]:
jac = jac_fn(x=z)

In [30]:
jac.shape

torch.Size([1, 525, 64])

In [31]:
from einops import rearrange

gmx = jac @ rearrange(jac, "b a d -> b d a")
gmx.shape

torch.Size([1, 525, 525])

In [32]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

sim = gmx[0].detach().numpy()  # (525, 525)


def cosine_sim_matrix(X, eps=1e-12):
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)
    return X @ X.T


def mutual_knn_edges(S, k=3, min_sim=None):
    """Returns list of (i, j, sim) edges using mutual kNN."""
    L = S.shape[0]
    nn = []
    for i in range(L):
        row = S[i].copy()
        row[i] = -np.inf
        idx = np.argpartition(row, -k)[-k:]
        idx = idx[np.argsort(row[idx])[::-1]]
        nn.append(set(idx.tolist()))

    edges = []
    for i in range(L):
        for j in nn[i]:
            if i in nn[j]:  # mutual
                if i < j:
                    s = float(S[i, j])
                    if min_sim is None or s >= min_sim:
                        edges.append((i, j, s))
    return edges


def clustered_order(S):
    """Simple greedy ordering by similarity for block visibility."""
    L = S.shape[0]
    remaining = set(range(L))
    order = []
    cur = 0
    order.append(cur)
    remaining.remove(cur)
    while remaining:
        nxt = max(remaining, key=lambda j: S[cur, j])
        order.append(nxt)
        remaining.remove(nxt)
        cur = nxt
    return np.array(order)


def plot_heatmap(
    S,
    order=None,
    title="Similarity heatmap",
    n_positions=25,
    alphabet="ACDEFGHIKLMNPQRSTVWY_",
):
    """Plot heatmap with position dividers and per-position AA tick labels."""
    S2 = S[order][:, order] if order is not None else S
    n_aa = len(alphabet)
    N = S2.shape[0]

    # tick labels: "0:A", "0:C", ...
    tickvals = list(range(N))
    ticktext = [f"{i // n_aa}:{alphabet[i % n_aa]}" for i in range(N)]

    fig = px.imshow(
        S2,
        origin="lower",
        aspect="auto",
        labels=dict(color="Similarity"),
        title=title,
    )

    # position divider lines
    for p in range(1, n_positions):
        line_pos = p * n_aa - 0.5
        fig.add_hline(y=line_pos, line_width=1, line_color="white", opacity=0.6)
        fig.add_vline(x=line_pos, line_width=1, line_color="white", opacity=0.6)

    fig.update_xaxes(
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=90,
        tickfont=dict(size=5),
        title="Position : AA",
    )
    fig.update_yaxes(
        tickvals=tickvals,
        ticktext=ticktext,
        tickfont=dict(size=5),
        title="Position : AA",
    )

    fig.update_layout(height=900, width=950)
    fig.show()


def plot_sparse_edge_scatter(
    edges,
    title="Sparse similarity graph (edge list view)",
    n_positions=25,
    alphabet="ACDEFGHIKLMNPQRSTVWY_",
):
    """Show edges in (i,j) space colored by sim — readable at large L."""
    if not edges:
        print("No edges after filtering.")
        return
    n_aa = len(alphabet)
    i = [e[0] for e in edges]
    j = [e[1] for e in edges]
    s = [e[2] for e in edges]
    fig = go.Figure(
        go.Scattergl(
            x=i,
            y=j,
            mode="markers",
            marker=dict(
                size=5,
                color=s,
                cmin=0,
                cmax=1,
                colorbar=dict(title="sim"),
            ),
            text=[
                f"{a // n_aa}:{alphabet[a % n_aa]} – {b // n_aa}:{alphabet[b % n_aa]}<br>sim={v:.3f}"
                for a, b, v in edges
            ],
            hoverinfo="text",
        )
    )

    # position divider lines
    N = n_positions * n_aa
    for p in range(1, n_positions):
        line_pos = p * n_aa - 0.5
        fig.add_hline(y=line_pos, line_width=0.5, line_color="gray", opacity=0.4)
        fig.add_vline(x=line_pos, line_width=0.5, line_color="gray", opacity=0.4)

    # tick labels: "pos:AA"
    tickvals = list(range(N))
    ticktext = [f"{i // n_aa}:{alphabet[i % n_aa]}" for i in range(N)]

    fig.update_xaxes(
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=90,
        tickfont=dict(size=5),
        title="Position : AA",
    )
    fig.update_yaxes(
        tickvals=tickvals,
        ticktext=ticktext,
        tickfont=dict(size=5),
        title="Position : AA",
    )

    fig.update_layout(
        title=title,
        height=900,
        width=950,
    )
    fig.show()


# --- reordered heatmap ---
S = cosine_sim_matrix(sim)
order = clustered_order(S)
plot_heatmap(S, order=order, title="Similarity (cosine), reordered")

# --- sparse mutual kNN edges ---
edges = mutual_knn_edges(
    S, k=3, min_sim=np.quantile(S[np.triu_indices(S.shape[0], 1)], 0.999)
)
plot_sparse_edge_scatter(edges, title="Mutual kNN edges (k=3) + top 0.1% threshold")

In [1]:
# import torch
# torch.linalg
# get_mutations_from_s_u_standard

In [ ]:
# PCOA